In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
from pyspark.sql import functions as F

In [0]:
catalog_name = 'ecommerce'

# Define schema for the data file
brand_schema = StructType([
    StructField("brand_code", StringType(), False),
    StructField("brand_name", StringType(), True),
    StructField("category_code", StringType(), True),
    ])

In [0]:
raw_data_path = "/Volumes/ecommerce/source_data/raw/brands/*.csv"
df = spark.read.option('header', 'true').option("delimeter", ",").schema(brand_schema).csv(raw_data_path)

# add metadata columns
df = df.withColumn('_source_file', F.col('_metadata.file_path')) \
    .withColumn('ingested_at', F.current_timestamp())
display(df.limit(10))

In [0]:
# write to table
df.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_brands")

**Category**

In [0]:
# Define schema for the data file
category_schema = StructType([
    StructField("category_code", StringType(), False),
    StructField("category_name", StringType(), True)
])

# Load data using the schema
raw_data_path = "/Volumes/ecommerce/source_data/raw/category/*.csv"
df_raw = spark.read.option('header', 'true').option("delimeter", ",").schema(category_schema).csv(raw_data_path)

# add metadata columns
df_raw = df_raw.withColumn('ingested_at', F.current_timestamp())\
    .withColumn('_source_file', F.col('_metadata.file_path'))
display(df_raw.limit(10))




In [0]:
# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_category)
df_raw.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_category")

**Products**

In [0]:
# Define schema for the data file
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),
    StructField("length_cm", StringType(), True),
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True),
    StructField("file_name", StringType(), True),
    StructField("ingest_timestamp", TimestampType(), True)
])

# Load data using the scema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/products/*.csv"
df_products = spark.read.option('header', 'true').option("delimeter", ",").schema(products_schema).csv(raw_data_path)\
    .withColumn('file_name', F.col('_metadata.file_path'))\
    .withColumn('ingest_timestamp', F.current_timestamp())
display(df_products.limit(10))




In [0]:
# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_products)
df_products.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_products")

**Customers**

In [0]:
# Define schema for the data file
customers_shema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])

In [0]:
# Load data using the scema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/customers/*.csv"
df_customers = spark.read.option('header', 'true').option("delimeter", ",").schema(customers_shema).csv(raw_data_path)\
    .withColumn('file_name', F.col('_metadata.file_path'))\
    .withColumn('ingest_timestamp', F.current_timestamp())
display(df_customers.limit(10))

In [0]:
# Write raw data to the Bronze layer (catalog: dbdemos, schema: ecommerce, table: customers_bronze)
df_customers.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_customers")

**Date**

In [0]:
# Define schema for the data file

date_schema = StructType([
    StructField("date", DateType(), False),
    StructField("year", IntegerType(), True),
    StructField("day_name", StringType(), True),
    StructField("quarter", IntegerType(), True),
    StructField("week_of_year", IntegerType(), True)
])

In [0]:
# Load data using the scema defined 
raw_data_path = "/Volumes/ecommerce/source_data/raw/date/*.csv"
df_date = spark.read.option('header', 'true').option("delimeter", ",").schema(date_schema).csv(raw_data_path)\
    .withColumn('date', F.col('_metadata.file_path'))\
    .withColumn('ingest_timestamp', F.current_timestamp())
display(df_date.limit(10))


In [0]:
# Write raw data to the Bronze layer (catalog: dbdemos, schema: ecommerce, table: date_bronze)
df_date.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.bronze.brz_date")